# Unit 5: mne-sample-data 完整流程综合案例

## 概述

本单元整合前四个单元的知识，使用 **mne-sample-data** 数据集完成从原始数据加载到ERP和时频分析的完整EEG数据处理流程。

## 数据集说明

**mne-sample-data** 包含：
- 一名受试者的MEG/EEG联合记录
- 听觉和视觉诱发的oddball实验范式
- 四种实验条件：
  - Auditory Left (左耳听觉刺激)
  - Auditory Right (右耳听觉刺激)
  - Visual Left (左侧视野视觉刺激)
  - Visual Right (右侧视野视觉刺激)

## 完整分析流程

```
1. 数据加载
   ↓
2. 数据探索与质量控制
   ↓
3. 预处理
   ├── 滤波
   ├── 重参考
   ├── 坏道检测与插值
   └── ICA去伪迹
   ↓
4. Epoch提取与基线校正
   ↓
5. 伪迹拒绝
   ↓
6. ERP分析
   ├── Evoked计算
   ├── 成分测量
   ├── 可视化
   └── 统计检验
   ↓
7. 时频分析
   ├── Morlet小波变换
   ├── 基线校正
   └── ERS/ERD可视化
   ↓
8. 结果报告
```


## 实践代码

In [ ]:
# 导入所有必要的库
import mne
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.signal import find_peaks
from mne.preprocessing import ICA
from mne.time_frequency import tfr_morlet

# 设置MNE日志级别
mne.set_log_level('WARNING')

# 设置绘图风格
plt.style.use('default')

print("=" * 60)
print("Unit 5: mne-sample-data 完整流程综合案例")
print("=" * 60)

### 阶段 1: 数据加载与探索

In [ ]:
# 1.1 加载mne-sample-data
print("【步骤 1】数据加载")
print("-" * 40)

# 下载并获取示例数据路径
data_path = mne.datasets.sample.data_path()
raw_fname = data_path / 'MEG' / 'sample' / 'sample_audvis_filt-0-40_raw.fif'

# 读取原始数据
raw = mne.io.read_raw_fif(raw_fname, preload=True)

print(f"文件: {raw_fname.name}")
print(f"采样率: {raw.info['sfreq']} Hz")
print(f"持续时间: {raw.times[-1]:.1f} 秒 ({raw.times[-1]/60:.1f} 分钟)")
print(f"总通道数: {len(raw.ch_names)}")

# 查看通道类型分布
ch_type_counts = {}
for ch in raw.info['chs']:
    ch_type = mne.io.meas_info.channel_type(raw.info, ch['chan'])
    ch_type_counts[ch_type] = ch_type_counts.get(ch_type, 0) + 1

print(f"\n通道类型分布:")
for ch_type, count in sorted(ch_type_counts.items()):
    print(f"  {ch_type}: {count}")

In [ ]:
# 1.2 数据可视化探索
print("【步骤 2】数据探索")
print("-" * 40)

# 绘制原始数据（前30秒）
raw.plot(duration=30, n_channels=40, scalings='auto',
         title='Raw MEG/EEG Data (First 30 seconds)',
         block=False)

# 绘制EEG通道位置
montage = raw.get_montage()
if montage:
    montage.plot(kind='topomap', show_names=True, 
                 title='EEG Electrode Layout')

In [ ]:
# 1.3 事件检测
events = mne.find_events(raw, stim_channel='STI 014', verbose=False)

print(f"检测到事件总数: {len(events)}")
print(f"\n事件类型分布:")
event_ids, counts = np.unique(events[:, 2], return_counts=True)

event_dict = {
    'auditory/left': 1,
    'auditory/right': 2,
    'visual/left': 3,
    'visual/right': 4
}

for event_id, label in event_dict.items():
    count = counts[event_ids == label][0] if label in event_ids else 0
    print(f"  {event_id} (ID={label}): {count} trials")

### 阶段 2: 数据预处理

In [ ]:
# 2.1 选择EEG通道
print("【步骤 3】预处理 - 通道选择")
print("-" * 40)

# 仅保留EEG和相关通道
raw_eeg = raw.copy().pick_types(eeg=True, stim=True, eog=True)
print(f"选择后通道数: {len(raw_eeg.ch_names)}")
print(f"EEG通道: {len([ch for ch in raw_eeg.ch_names if ch.startswith('EEG')])}")

In [ ]:
# 2.2 滤波
print("\n【步骤 4】预处理 - 滤波")
print("-" * 40)

# 带通滤波：0.1-40 Hz
# 适用于ERP分析
raw_eeg.filter(l_freq=0.1, h_freq=40, fir_design='firwin', verbose=False)
print("带通滤波完成: 0.1-40 Hz")

# 显示滤波后的PSD
fig, ax = plt.subplots(figsize=(10, 5))
psds, freqs = mne.time_frequency.psd_welch(raw_eeg, fmin=0.1, fmax=50, 
                                            n_fft=2048, verbose=False)
psds_eeg = psds[[i for i, ch in enumerate(raw_eeg.ch_names) if ch.startswith('EEG')]]
mean_psd = 10 * np.log10(psds_eeg.mean(axis=0))
ax.plot(freqs, mean_psd, color='blue', linewidth=2)
ax.set_xlabel('Frequency (Hz)', fontsize=11)
ax.set_ylabel('Power (dB)', fontsize=11)
ax.set_title('PSD After Band-pass Filter (0.1-40 Hz)', fontsize=12)
ax.axvspan(0.1, 40, alpha=0.2, color='green', label='Pass band')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 2.3 重参考
print("\n【步骤 5】预处理 - 重参考")
print("-" * 40)

raw_eeg.set_eeg_reference(ref_channels='average', projection=True, verbose=False)
print("平均参考设置完成")

In [ ]:
# 2.4 坏道检测
print("\n【步骤 6】预处理 - 坏道检测")
print("-" * 40)

# 可视化检测坏道
raw_eeg.plot(duration=5, n_channels=60, title='Bad Channel Detection',
             block=False)
print("提示: 在可视化窗口中标记坏道后关闭窗口")
print(f"当前坏道: {raw_eeg.info['bads']}")

# 如果有坏道，进行插值
if len(raw_eeg.info['bads']) > 0:
    print(f"\n对 {len(raw_eeg.info['bads'])} 个坏道进行插值...")
    raw_eeg.interpolate_bads(reset_bads=True, verbose=False)
    print("坏道插值完成")
else:
    print("\n无坏道，跳过插值步骤")

In [ ]:
# 2.5 ICA去伪迹
print("\n【步骤 7】预处理 - ICA去伪迹")
print("-" * 40)

# 为ICA创建专用数据副本（需要高通滤波≥1 Hz）
raw_ica = raw_eeg.copy().filter(l_freq=1.0, h_freq=None, verbose=False)

# 配置并运行ICA
ica = ICA(n_components=0.95, method='fastica', max_iter='auto', 
          random_state=42)

print("拟合ICA模型...")
ica.fit(raw_ica, verbose=False)
print(f"ICA完成，提取了 {ica.n_components_} 个成分")

# 自动检测EOG相关成分
eog_indices, eog_scores = ica.find_bads_eog(raw_ica, verbose=False)
print(f"检测到的EOG成分: {eog_indices}")

# 排除伪迹成分
ica.exclude = eog_indices

# 应用ICA
raw_clean = raw_eeg.copy()
ica.apply(raw_clean, verbose=False)
print("ICA伪迹去除完成")

### 阶段 3: Epoch提取与伪迹拒绝

In [ ]:
# 3.1 创建Epochs
print("【步骤 8】Epoch提取")
print("-" * 40)

events = mne.find_events(raw_clean, stim_channel='STI 014', verbose=False)

tmin, tmax = -0.2, 0.8
epochs = mne.Epochs(raw_clean, events, event_id=event_dict,
                    tmin=tmin, tmax=tmax,
                    baseline=(-0.2, 0),
                    reject=dict(eeg=150e-6),
                    preload=True, verbose=False)

print(f"总事件数: {len(events)}")
print(f"保留的epochs: {len(epochs)}")
print(f"拒绝的epochs: {len(events) - len(epochs)}")

print(f"\n各条件trial数量:")
for cond in epochs.event_id.keys():
    print(f"  {cond}: {len(epochs[cond])}")

### 阶段 4: ERP分析

In [ ]:
# 4.1 计算Evoked
print("【步骤 9】ERP分析 - Evoked计算")
print("-" * 40)

# 按条件计算平均ERP
evoked_aud_l = epochs['auditory/left'].average()
evoked_aud_r = epochs['auditory/right'].average()
evoked_vis_l = epochs['visual/left'].average()
evoked_vis_r = epochs['visual/right'].average()

# 合并感觉模态
evoked_aud = mne.combine_evoked([evoked_aud_l, evoked_aud_r], weights='nave')
evoked_vis = mne.combine_evoked([evoked_vis_l, evoked_vis_r], weights='nave')

print(f"听觉平均: {evoked_aud.nave} trials")
print(f"视觉平均: {evoked_vis.nave} trials")

# 绘制ERP对比
fig, ax = plt.subplots(figsize=(10, 5))
ch_pick = 'EEG 053'
ch_idx = evoked_aud.ch_names.index(ch_pick)

ax.plot(evoked_aud.times, evoked_aud.data[ch_idx] * 1e6, 
        'b-', linewidth=2, label='Auditory')
ax.plot(evoked_vis.times, evoked_vis.data[ch_idx] * 1e6, 
        'r-', linewidth=2, label='Visual')

ax.axhline(0, color='k', linewidth=0.5)
ax.axvline(0, color='gray', linestyle='--', linewidth=0.5)
ax.set_xlabel('Time (s)', fontsize=11)
ax.set_ylabel('Amplitude (μV)', fontsize=11)
ax.set_title(f'ERP Comparison at {ch_pick}', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 4.2 Joint Plot可视化
print("\n【步骤 10】ERP分析 - Joint Plot")
print("-" * 40)

evoked_aud.plot_joint(times=[0.1, 0.17, 0.3],
                      title='Auditory Evoked Response',
                      ts_args={'xlim': [-0.2, 0.8]})
plt.show()

In [ ]:
# 4.3 ERP成分测量
print("\n【步骤 11】ERP分析 - 成分测量")
print("-" * 40)

def measure_component(evoked, ch_name, t_window, name):
    ch_idx = evoked.ch_names.index(ch_name)
    tmask = (evoked.times >= t_window[0]) & (evoked.times <= t_window[1])
    data = evoked.data[ch_idx, tmask] * 1e6
    times = evoked.times[tmask]
    peak_idx = np.argmax(np.abs(data))
    return {
        'name': name,
        'amplitude': data[peak_idx],
        'latency': times[peak_idx] * 1000
    }

ch_target = 'EEG 053'
components_to_measure = [
    ('P1', (0.05, 0.12)),
    ('N1', (0.10, 0.18)),
    ('P2', (0.15, 0.22)),
    ('P3', (0.25, 0.45))
]

print(f"\nERP成分测量 - 通道: {ch_target}")
print(f"{'成分':<8} {'听觉幅度(μV)':<16} {'听觉潜伏期(ms)':<18} "
      f"{'视觉幅度(μV)':<16} {'视觉潜伏期(ms)':<16}")
print("-" * 74)

results = {}
for comp_name, t_win in components_to_measure:
    aud_res = measure_component(evoked_aud, ch_target, t_win, comp_name)
    vis_res = measure_component(evoked_vis, ch_target, t_win, comp_name)
    results[comp_name] = {'auditory': aud_res, 'visual': vis_res}
    print(f"{comp_name:<8} {aud_res['amplitude']:>12.2f}      "
          f"{aud_res['latency']:>10.1f}      "
          f"{vis_res['amplitude']:>12.2f}      "
          f"{vis_res['latency']:>10.1f}")

In [ ]:
# 4.4 统计分析
print("\n【步骤 12】ERP分析 - 统计检验")
print("-" * 40)

p3_window = (0.3, 0.5)
tmin_idx = np.searchsorted(epochs.times, p3_window[0])
tmax_idx = np.searchsorted(epochs.times, p3_window[1])
ch_idx = epochs.ch_names.index(ch_target)

aud_amps = np.mean(epochs['auditory'].get_data()[:, ch_idx, tmin_idx:tmax_idx], axis=1) * 1e6
vis_amps = np.mean(epochs['visual'].get_data()[:, ch_idx, tmin_idx:tmax_idx], axis=1) * 1e6

min_n = min(len(aud_amps), len(vis_amps))
t_stat, p_value = stats.ttest_rel(aud_amps[:min_n], vis_amps[:min_n])

print(f"\nP3窗口 ({p3_window[0]*1000:.0f}-{p3_window[1]*1000:.0f} ms) 统计检验")
print(f"通道: {ch_target}")
print(f"听觉平均: {np.mean(aud_amps):.2f} μV (SD={np.std(aud_amps):.2f})")
print(f"视觉平均: {np.mean(vis_amps):.2f} μV (SD={np.std(vis_amps):.2f})")
print(f"t({min_n-1}) = {t_stat:.3f}, p = {p_value:.4f}")

sig = "***" if p_value < 0.001 else "**" if p_value < 0.01 else "*" if p_value < 0.05 else "n.s."
print(f"显著性: {sig}")

# 可视化统计结果
fig, ax = plt.subplots(figsize=(6, 5))
bp = ax.boxplot([aud_amps, vis_amps], labels=['Auditory', 'Visual'],
                patch_artist=True, widths=0.5)
for patch, color in zip(bp['boxes'], ['lightblue', 'lightcoral']):
    patch.set_facecolor(color)
ax.set_ylabel(f'P3 Mean Amplitude (μV)', fontsize=11)
ax.set_title(f'Auditory vs Visual: P3 Window\n{ch_target}', fontsize=12)
ax.grid(True, alpha=0.3, axis='y')

y_max = max(np.max(aud_amps), np.max(vis_amps))
ax.plot([0.5, 0.5, 1.5, 1.5], [y_max*1.1, y_max*1.15, y_max*1.15, y_max*1.1],
        color='black', linewidth=1)
ax.text(1, y_max*1.18, sig, ha='center', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 阶段 5: 时频分析

In [ ]:
# 5.1 重新创建适合时频分析的epochs
print("【步骤 13】时频分析 - 数据准备")
print("-" * 40)

events = mne.find_events(raw_clean, stim_channel='STI 014', verbose=False)
epochs_tf = mne.Epochs(raw_clean, events, event_id=event_dict,
                       tmin=-1.0, tmax=1.0,
                       baseline=None,
                       reject=dict(eeg=150e-6),
                       preload=True, verbose=False)

epochs_tf_aud = epochs_tf['auditory'].copy()
print(f"时频分析epochs: {len(epochs_tf_aud)} trials")
print(f"时间窗口: [{epochs_tf_aud.tmin}, {epochs_tf_aud.tmax}] s")

In [ ]:
# 5.2 Morlet小波时频分析
print("\n【步骤 14】时频分析 - Morlet小波变换")
print("-" * 40)

freqs = np.arange(3, 40, 2)
n_cycles = freqs / 2.0

print(f"频率范围: {freqs[0]}-{freqs[-1]} Hz")
print("正在计算时频变换...")

power = tfr_morlet(epochs_tf_aud, freqs=freqs, n_cycles=n_cycles,
                   use_fft=True, return_itc=False, average=True, verbose=False)

# 基线校正
power.apply_baseline(baseline=(-0.5, -0.1), mode='logratio')
print("时频计算与基线校正完成")

In [ ]:
# 5.3 时频可视化
print("\n【步骤 15】时频分析 - 可视化")
print("-" * 40)

ch_tf = 'EEG 053'
ch_tf_idx = power.ch_names.index(ch_tf)

# 时频图
fig, ax = plt.subplots(figsize=(12, 6))
power.plot(picks=[ch_tf_idx], show=False, axes=ax, cmap='RdBu_r',
           vmin=-1.0, vmax=1.0, colorbar=True,
           title=f'Time-Frequency Power (Auditory, {ch_tf})')
ax.axvline(0, color='white', linestyle='--', linewidth=2, alpha=0.8)
plt.tight_layout()
plt.show()

# 频段时程
fig, ax = plt.subplots(figsize=(10, 5))
bands = {'Theta': (4, 8), 'Alpha': (8, 13), 'Beta': (13, 30)}
colors = {'Theta': 'purple', 'Alpha': 'green', 'Beta': 'blue'}

for band, (fmin, fmax) in bands.items():
    fmask = (freqs >= fmin) & (freqs <= fmax)
    band_ts = np.mean(power.data[ch_tf_idx, fmask, :], axis=0)
    ax.plot(power.times, band_ts, color=colors[band], linewidth=2,
            label=f'{band} ({fmin}-{fmax} Hz)')

ax.axhline(0, color='black', linewidth=0.5)
ax.axvline(0, color='gray', linestyle='--', linewidth=0.5)
ax.set_xlabel('Time (s)', fontsize=11)
ax.set_ylabel('Power Change (dB)', fontsize=11)
ax.set_title(f'Band-specific Power Time Course ({ch_tf})', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 阶段 6: 综合报告

In [ ]:
# 6.1 生成完整分析报告
print("\n" + "=" * 60)
print("完 整 分 析 报 告")
print("=" * 60)

print("\n【数据集信息】")
print(f"- 数据文件: {raw_fname.name}")
print(f"- 采样率: {raw.info['sfreq']} Hz")
print(f"- 记录时长: {raw.times[-1]:.1f} 秒")
print(f"- EEG通道数: {len([ch for ch in raw_eeg.ch_names if ch.startswith('EEG')])}")

print("\n【预处理参数】")
print(f"- 带通滤波: 0.1-40 Hz (FIR)")
print(f"- 参考方式: 平均参考")
print(f"- ICA成分数: {ica.n_components_}")
print(f"- 排除的ICA成分: {ica.exclude}")
print(f"- Epoch窗口: [{tmin}, {tmax}] s")
print(f"- 基线窗口: [{tmin}, 0] s")
print(f"- 幅度拒绝阈值: ±150 μV")

print("\n【Trial统计】")
print(f"- 总事件数: {len(events)}")
print(f"- 保留epochs: {len(epochs)}")
for cond in epochs.event_id.keys():
    print(f"  {cond}: {len(epochs[cond])} trials")

print("\n【ERP分析结果】")
print(f"目标通道: {ch_target}")
for comp_name in components_to_measure:
    comp = comp_name[0]
    aud = results[comp]['auditory']
    vis = results[comp]['visual']
    print(f"  {comp}: 听觉 {aud['amplitude']:.2f} μV @ {aud['latency']:.0f} ms | "
          f"视觉 {vis['amplitude']:.2f} μV @ {vis['latency']:.0f} ms")

print(f"\n【统计检验结果】")
print(f"- P3窗口: {p3_window[0]*1000:.0f}-{p3_window[1]*1000:.0f} ms")
print(f"- t({min_n-1}) = {t_stat:.3f}, p = {p_value:.4f} {sig}")

print("\n【时频分析参数】")
print(f"- 方法: Morlet小波变换")
print(f"- 频率范围: {freqs[0]}-{freqs[-1]} Hz ({len(freqs)} points)")
print(f"- 周期数: {n_cycles[0]:.1f}-{n_cycles[-1]:.1f}")
print(f"- 基线校正: logratio, window=(-0.5, -0.1) s")

print("\n" + "=" * 60)
print("分析完成!")
print("=" * 60)

In [ ]:
# 6.2 保存结果
print("\n【步骤 17】保存结果")
print("-" * 40)

# 保存预处理后的epochs
epochs.save('final_epochs-epo.fif', overwrite=True)
print(f"Epochs保存: final_epochs-epo.fif")

# 保存Evoked数据
evoked_aud.save('auditory-evoked-ave.fif', overwrite=True)
evoked_vis.save('visual-evoked-ave.fif', overwrite=True)
print(f"Evoked保存: auditory-evoked-ave.fif, visual-evoked-ave.fif")

# 保存时频数据
power.save('tfr-power-ave.fif', overwrite=True)
print(f"时频数据保存: tfr-power-ave.fif")

print("\n所有结果已保存!")

## 单元小结

**完整流程回顾：**
1. **数据加载与探索**：了解数据结构和质量
2. **预处理**：滤波、重参考、坏道处理、ICA去伪迹
3. **Epoch提取**：事件分段、基线校正、伪迹拒绝
4. **ERP分析**：Evoked计算、成分测量、统计检验
5. **时频分析**：Morlet小波、基线校正、ERS/ERD
6. **结果报告**：整合所有分析结果

**关键学习点：**
- 预处理是后续分析的基础，每一步都需谨慎
- ERP和时频分析提供互补的信息
- 统计分析应考虑多重比较问题
- 结果的可重现性需要良好的代码管理和参数记录

**进一步学习建议：**
- 尝试使用BIDS格式组织数据
- 探索源定位分析（Source Localization）
- 学习多受试者组水平的统计分析
- 尝试连接分析（Connectivity Analysis）
- 使用MNE-BIDS进行数据共享